In [3]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.losses import QLIKELoss
from src.eval.scorer import VolatilityScorer
from src.models.wtd_har_gru_wrapper import CausalWTDHARWindowDataset, WTDHARGRUModel
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "wtd_har_gru"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any()
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any()

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


In [6]:
import time

scorer = VolatilityScorer()

best_model = WTDHARGRUModel(
    look_back=14,
    lag_weekly=5,
    lag_monthly=22,
    wavelet="db4",
    wtd_level=2,
    hidden_size=64,
    num_layers=1,
    dropout=0.2,
    learning_rate=1e-3,
    weight_decay=1e-6,
    max_epochs=200,
    patience=20,
    batch_size=128,
    retrain_freq=1,
    seed=42,
)

start_time = time.time()
best_model.fit(y_train, rs_train)
end_time = time.time()

elapsed_time = end_time - start_time
print(f">> fit OK en {elapsed_time:.2f} s")
print("   diagnostics:", best_model.get_diagnostics())

>> fit OK en 13.47 s
   diagnostics: {'converged': True, 'epochs_run': 25}


In [6]:
df_bigru

,look_back,lag_weekly,lag_monthly,wavelet,wtd_level,hidden_size,num_layers,dropout,learning_rate,weight_decay,...,seed,fold,train_size,val_size,converged,epochs_run,train_time_sec,qlike,lr,wd
0,28,7,22,sym4,1,128,1,0.285058,0.000169,0.000024,...,42,1,1209,192,True,48,9.04,0.333099,0.0002,2.4e-05
1,28,7,22,sym4,1,128,1,0.285058,0.000169,0.000024,...,42,2,1401,192,True,40,1.96,1.649727,0.0002,2.4e-05
2,28,7,22,sym4,1,128,1,0.285058,0.000169,0.000024,...,42,3,1593,192,True,33,1.84,0.405364,0.0002,2.4e-05
3,28,7,22,sym4,1,128,1,0.285058,0.000169,0.000024,...,42,4,1785,192,True,32,1.98,0.333218,0.0002,2.4e-05
4,28,7,22,sym4,1,128,1,0.285058,0.000169,0.000024,...,42,5,1977,193,True,36,2.44,0.805299,0.0002,2.4e-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,14,7,30,sym4,1,32,1,0.018552,0.000108,0.000929,...,42,4,1785,192,True,145,7.75,0.349577,0.0001,9.3e-04
64,14,7,30,sym4,1,32,1,0.018552,0.000108,0.000929,...,42,5,1977,193,True,171,9.75,0.574156,0.0001,9.3e-04
65,21,7,30,sym4,1,64,1,0.073213,0.000232,0.000737,...,42,1,1209,192,True,79,32.67,0.334488,0.0002,7.4e-04
66,21,7,30,sym4,1,64,1,0.073213,0.000232,0.000737,...,42,2,1401,192,True,53,17.89,0.617710,0.0002,7.4e-04


In [7]:
exec_dir = RESULTS_PATH / "exec_1"

df_wtd = pd.read_csv(exec_dir / "WTDHARGRUModel_results.csv")

df_wtd["lr"] = df_wtd["learning_rate"].apply(lambda x: f"{x:.4f}")
df_wtd["wd"] = df_wtd["weight_decay"].apply(lambda x: f"{x:.1e}")
df_wtd["dropout_fmt"] = df_wtd["dropout"].apply(lambda x: f"{x:.3f}")

cols_agrupar = [
    "look_back",
    "lag_weekly",
    "lag_monthly",
    "wavelet",
    "wtd_level",
    "hidden_size",
    "num_layers",
    "dropout_fmt",
    "lr",
    "wd",
]

df_agg = (
    df_wtd.groupby(cols_agrupar, dropna=False)["qlike"]
    .agg(["mean", "std"])
    .reset_index()
)
df_agg["QLIKE"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)

df_top5 = df_agg.sort_values(by="mean").head(5).rename(
    columns={
        "look_back": "Lookback",
        "lag_weekly": "Lag Sem.",
        "lag_monthly": "Lag Mes.",
        "wavelet": "Wavelet",
        "wtd_level": "Nivel WTD",
        "hidden_size": "Neuronas",
        "num_layers": "Capas",
        "dropout_fmt": "Dropout",
        "lr": "LR",
        "wd": "Weight Decay",
    }
)

df_final = df_top5[
    [
        "Lookback",
        "Lag Sem.",
        "Lag Mes.",
        "Wavelet",
        "Nivel WTD",
        "Neuronas",
        "Capas",
        "Dropout",
        "LR",
        "Weight Decay",
        "QLIKE",
    ]
]

tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,Lag Sem.,Lag Mes.,Wavelet,Nivel WTD,Neuronas,Capas,Dropout,LR,Weight Decay,QLIKE
21,7,30,sym4,1,64,1,0.073,0.0002,7.4e-04,0.453 ± 0.147
14,7,30,sym4,1,64,1,0.011,0.0003,6.7e-04,0.478 ± 0.179
14,7,30,sym4,1,64,1,0.072,0.0002,4.9e-04,0.490 ± 0.200
14,7,30,sym4,1,32,1,0.019,0.0001,9.3e-04,0.512 ± 0.241
14,5,30,sym4,1,64,1,0.079,0.0003,1.2e-04,0.532 ± 0.262


In [8]:
import time

scorer = VolatilityScorer()

best_model = WTDHARGRUModel(
    look_back=21,
    lag_weekly=7,
    lag_monthly=30,
    wavelet="sym4",
    wtd_level=1,
    hidden_size=64,
    num_layers=1,
    dropout=0.073,
    learning_rate=0.0002,
    weight_decay=7.4e-4,
    max_epochs=200,
    patience=20,
    batch_size=128,
    retrain_freq=1,
    seed=42,
)

start_time = time.time()
best_model.fit(y_train, rs_train)
end_time = time.time()

elapsed_time = end_time - start_time
print(f">> fit OK en {elapsed_time:.2f} s")
print("   diagnostics:", best_model.get_diagnostics())

>> fit OK en 51.58 s
   diagnostics: {'converged': True, 'epochs_run': 42}


In [9]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from src.models.wtd_har_gru_wrapper import CausalWTDHARWindowDataset
from src.utils import test_toolkit as ts

best_model.net.eval()
look_back = best_model.context_length

feats_train, rs_train_scaled_target = best_model.causal_features(y_train, rs_train)

train_eval_dataset = CausalWTDHARWindowDataset(
    features=feats_train,
    targets=rs_train_scaled_target,
    look_back=look_back,
)
train_eval_loader = DataLoader(
    train_eval_dataset, batch_size=best_model.batch_size, shuffle=False
)

preds_train_list = []
with torch.no_grad():
    for x_batch, _ in train_eval_loader:
        x_batch = x_batch.to(best_model.device)
        preds_batch = best_model.net(x_batch).squeeze(-1).cpu().numpy()
        preds_train_list.append(preds_batch)

preds_train_var = np.concatenate(preds_train_list) / best_model.target_scale
n_preds = len(preds_train_var)

dates_aligned = df_train.index[look_back : look_back + n_preds]
rs_volatilidad_aligned = np.sqrt(
    np.maximum(rs_train[look_back : look_back + n_preds], 0.0)
)
pred_volatilidad = np.sqrt(np.maximum(preds_train_var, 0.0))

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,
    pred_variance=pred_volatilidad,
    title="Ajuste dentro de la Muestra de Entrenamiento (WTD-HAR-GRU)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad WTD-HAR-GRU $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

In [ ]:
import optuna

from src.models.wtd_har_gru_wrapper import WTDHARGRUModel


def wtd_har_gru_search_space(trial: optuna.Trial) -> dict:
    return {
        "look_back": trial.suggest_categorical("look_back", [7, 14, 21, 28]),
        "lag_weekly": trial.suggest_categorical("lag_weekly", [5, 7]),
        "lag_monthly": trial.suggest_categorical("lag_monthly", [22, 30]),
        "wavelet": "db4",
        "wtd_level": trial.suggest_categorical("wtd_level", [1, 2, 3]),
        "hidden_size": trial.suggest_categorical("hidden_size", [32, 64, 128]),
        "num_layers": trial.suggest_int("num_layers", 1, 2),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42,
    }


exec_id = 1
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=WTDHARGRUModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5,
    warmup=current_warmup,
    embargo=1,
    n_trials=50,
    suggest_params_fn=wtd_har_gru_search_space,
    early_stop_rounds=15,
)

trainer.run(y_train, rs_train)

In [5]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
exec_dir = PROJECT_ROOT / "results" / "wtd_har_gru" / "exec_1"

print("exec_dir:", exec_dir.resolve(), "| existe:", exec_dir.exists())
if exec_dir.exists():
    print("Archivos en exec_1:", list(exec_dir.rglob("*")))

exec_dir: C:\Users\jcami\Documents\brent-crude-nlp-market-dynamics\results\wtd_har_gru\exec_1 | existe: True
Archivos en exec_1: [WindowsPath('C:/Users/jcami/Documents/brent-crude-nlp-market-dynamics/results/wtd_har_gru/exec_1/optuna_study.db'), WindowsPath('C:/Users/jcami/Documents/brent-crude-nlp-market-dynamics/results/wtd_har_gru/exec_1/WTDHARGRUModel_results.csv')]


In [8]:
from pathlib import Path
import shutil
import optuna
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
exec_dir = PROJECT_ROOT / "results" / "wtd_har_gru" / "exec_1"
backup_dir = exec_dir / "backup"
backup_dir.mkdir(exist_ok=True)

db_file = exec_dir / "optuna_study.db"
csv_file = exec_dir / "WTDHARGRUModel_results.csv"

# 1. Respaldo antes de modificar nada
shutil.copy2(db_file, backup_dir / "optuna_study.db.bak")
shutil.copy2(csv_file, backup_dir / "WTDHARGRUModel_results.csv.bak")

# 2. Marcar trial RUNNING como FAIL desde el storage de Optuna
storage_url = f"sqlite:///{db_file.resolve()}"
study = optuna.load_study(study_name="WTDHARGRUModel_study", storage=storage_url)

for t in study.trials:
    if t.state == optuna.trial.TrialState.RUNNING:
        print(f"Marcando Trial {t.number} (RUNNING) como FAIL...")
        study._storage.set_trial_state_values(
            t._trial_id,
            state=optuna.trial.TrialState.FAIL,
        )

# 3. Dejar en el CSV solo configuraciones con los 5 folds completos
df_csv = pd.read_csv(csv_file)
param_cols = [
    "look_back",
    "lag_weekly",
    "lag_monthly",
    "wavelet",
    "wtd_level",
    "hidden_size",
    "num_layers",
    "dropout",
    "learning_rate",
    "weight_decay",
    "max_epochs",
    "patience",
    "retrain_freq",
    "seed",
]

fold_counts = df_csv.groupby(param_cols, dropna=False)["fold"].transform("count")
df_clean = df_csv[fold_counts == 5].copy()
df_clean.to_csv(csv_file, index=False)

print(f"Filas antes: {len(df_csv)} | Filas limpias (5 folds c/u): {len(df_clean)}")
print("Trials completos en estudio:", len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]))
print("Mejor trial actual:", study.best_trial.number, "| Best QLIKE:", study.best_value)

Marcando Trial 13 (RUNNING) como FAIL...
Filas antes: 67 | Filas limpias (5 folds c/u): 65
Trials completos en estudio: 13
Mejor trial actual: 11 | Best QLIKE: 0.4783089311108334


In [9]:
import optuna

storage_url = f"sqlite:///{db_file.resolve()}"
study = optuna.load_study(study_name="WTDHARGRUModel_study", storage=storage_url)

n_trials_objetivo = 50  # o el número total que vayas a usar en WalkForwardTrainer
completos = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
fallidos = sum(t.state == optuna.trial.TrialState.FAIL for t in study.trials)
activos = sum(t.state == optuna.trial.TrialState.RUNNING for t in study.trials)

print(f"Total en DB: {len(study.trials)} | COMPLETE: {completos} | FAIL: {fallidos} | RUNNING: {activos}")
print(f"Trials nuevos que ejecutará trainer.run(n_trials={n_trials_objetivo}): {max(0, n_trials_objetivo - len(study.trials))}")

Total en DB: 14 | COMPLETE: 13 | FAIL: 1 | RUNNING: 0
Trials nuevos que ejecutará trainer.run(n_trials=50): 36


In [10]:
from pathlib import Path
import sqlite3

PROJECT_ROOT = Path.cwd().parent
db_file = PROJECT_ROOT / "results" / "wtd_har_gru" / "exec_1" / "optuna_study.db"

conn = sqlite3.connect(db_file)
cur = conn.cursor()

# Obtener el trial_id interno del trial con number = 13 en WTDHARGRUModel_study
rows = cur.execute(
    """
    SELECT t.trial_id
    FROM trials t
    JOIN studies s ON t.study_id = s.study_id
    WHERE s.study_name = ? AND t.number = ?
    """,
    ("WTDHARGRUModel_study", 13),
).fetchall()

trial_ids = [r[0] for r in rows]
print("trial_ids a borrar:", trial_ids)

for tid in trial_ids:
    for tbl in (
        "trial_params",
        "trial_values",
        "trial_user_attributes",
        "trial_system_attributes",
        "trial_intermediate_values",
        "trials",
    ):
        cur.execute(f"DELETE FROM {tbl} WHERE trial_id = ?", (tid,))

conn.commit()
conn.close()

trial_ids a borrar: [14]


In [11]:
from pathlib import Path
import optuna
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
exec_dir = PROJECT_ROOT / "results" / "wtd_har_gru" / "exec_1"

db_file = exec_dir / "optuna_study.db"
csv_file = exec_dir / "WTDHARGRUModel_results.csv"

storage_url = f"sqlite:///{db_file.resolve()}"
study = optuna.load_study(study_name="WTDHARGRUModel_study", storage=storage_url)
df_csv = pd.read_csv(csv_file)

completos = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
fallidos = sum(t.state == optuna.trial.TrialState.FAIL for t in study.trials)
activos = sum(t.state == optuna.trial.TrialState.RUNNING for t in study.trials)

print(f"DB -> total: {len(study.trials)} | COMPLETE: {completos} | FAIL: {fallidos} | RUNNING: {activos}")
print(f"CSV -> filas: {len(df_csv)} (esperado para 13 trials x 5 folds: 65)")

DB -> total: 13 | COMPLETE: 13 | FAIL: 0 | RUNNING: 0
CSV -> filas: 65 (esperado para 13 trials x 5 folds: 65)


In [ ]:
import optuna

from src.models.wtd_har_gru_wrapper import WTDHARGRUModel


def wtd_har_gru_search_space(trial: optuna.Trial) -> dict:
    return {
        "look_back": trial.suggest_categorical("look_back", [7, 14, 21, 28]),
        "lag_weekly": trial.suggest_categorical("lag_weekly", [5, 7]),
        "lag_monthly": trial.suggest_categorical("lag_monthly", [22, 30]),
        "wavelet": "sym4",
        "wtd_level": trial.suggest_categorical("wtd_level", [1, 2, 3]),
        "hidden_size": trial.suggest_categorical("hidden_size", [32, 64, 128]),
        "num_layers": trial.suggest_int("num_layers", 1, 2),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42,
    }


exec_id = 1
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

# Usa n_trials=14 para una prueba de 1 solo trial nuevo sobre los 13 existentes;
# cuando vayas por el total de 50, cámbialo a n_trials=50.
trainer = WalkForwardTrainer(
    model_class=WTDHARGRUModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5,
    warmup=current_warmup,
    embargo=1,
    n_trials=14,
    suggest_params_fn=wtd_har_gru_search_space,
    early_stop_rounds=15,
)

trainer.run(y_train, rs_train)

  0%|          | 0/1 [00:00<?, ?it/s]